# **PiCam GIF Generator v1**
#### This script reads in daily images and generates a GIF timelapse animation. Images were collected using camera-equipped Raspberry Pi computers (PiCams) through the RainManSR project. GIF anamation shows seasonal vegetation growth at the plot scale
#### Created 2021-08-26
#### Last modified: 2021-08-27
#### Charlie Devine (cjdevine@email.arizona.edu)
---------------------------------------------
#### ***For now, in order for this script to run properly it must be located within the same directory as the daily PiCam images. Daily images must first be copied/pasted from a 30-minute PiCam image set.***

---------------------------------------------
#### 1) Import libraries

In [ ]:
from PIL import Image, ImageDraw, ImageFont
import glob
import os
import numpy as np
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, PillowWriter
from matplotlib.dates import DateFormatter
from IPython.display import HTML

#### 2) Set filepaths

In [ ]:
os.chdir('..')

In [ ]:
# Set House and PiCam vars
house = 'House_1'
picam = '4_PiCam18'

# Set working directory
git_fp = os.getcwd()
images_fp = os.path.join(git_fp, 'Data', 'Images', house, picam, 'Images_for_GIFs')
gif_fp = os.path.join(git_fp, 'Figures_and_animations')

# Set GIF output filename/path
gif_filename = house + '_' + picam + '_2020_ImageTimelapse.gif'
gif_filepath = os.path.join(gif_fp, gif_filename)

# Set working directory
os.chdir(images_fp)

#### 3) Specify plot names visible in PiCam images (left to right)

In [ ]:
plotR = 'H1P8W1S4'
plotL = 'H1P7W2S1'
plotR_treatment = 'S4'
plotL_treatment = 'S1'

#### 4) Get list of filenames, extract date information

In [ ]:
img_filenames = glob.glob("*.jpg")
nfiles = len(img_filenames)

year = [''] * nfiles
month = [''] * nfiles
day = [''] * nfiles

for i in range(0,nfiles,1):
    filename = img_filenames[i]
    year[i] = filename[11:15]
    month[i] = filename[16:18]
    day[i] = filename[19:21]

dates_labs = [''] * nfiles
    
for i in range(0,nfiles,1):
    dates_labs[i] = year[i]+'-'+month[i]+'-'+day[i]

#### 5) If you want to crop the images to show just single plot, open an image and play around with the boudning box x/y values until you've cropped the image to the desired area.

In [ ]:
# im = Image.open(img_filenames[60])
# width, height = im.size
# print(width,height)
# x = 1000
# y = 0
# box = (0, y, width - x, height)
# im_cropped = im.crop(box)
# im_cropped.show()

#### 6) Create a function to overlay date text with the corresponding image frame.

In [ ]:
c = 255 # Color
inc = 10
fontsize = 200
textX = 800
textY = 1750

def create_image_with_text(image,text):
    global c, inc
    img = Image.open(image)
    draw = ImageDraw.Draw(img)
    draw.text((textX,textY), text, font = ImageFont.truetype("arial", fontsize), fill=(c,c,c))
    draw.text((525,1625),plotL_treatment, font = ImageFont.truetype("arial", 150), fill=(c,c,c))
    draw.text((1850,1625),plotR_treatment, font = ImageFont.truetype("arial", 150), fill=(c,c,c))
    c += inc
    return img

test = create_image_with_text(img_filenames[20],dates_labs[20])
test.show()

#### 7) Generate animation and save .gif file to directory.

In [ ]:
# Initialize frames
frames = []

for i in range(0,nfiles,1):
    frame_cropped_text = create_image_with_text(img_filenames[i],dates_labs[i])
    frames.append(frame_cropped_text)

# Save file
#output_filename = plotL+'_'+plotR+'_'+'Summer2020_DailyAnimation.gif'
output_filename = gif_filepath

frames[0].save(output_filename, 
               format = 'GIF',
               append_images = frames[1:],
               save_all = True,
               duration = 250, loop = 0)

#### 8) Read daily mean GCC from .txt files for left and right plots.

In [ ]:
gcc_fp = os.path.join(git_fp, 'Data', 'GCC', house)
os.chdir(gcc_fp)

In [ ]:
r_gcc = pd.read_csv('House1_Plot08_20200701_to_20201002_Daily_Means.txt', sep = r'\s+')
l_gcc = pd.read_csv('House1_Plot07_20200701_to_20201002_Daily_Means.txt', sep = r'\s+')

# Convert date fields
r_gcc['Date'] = pd.to_datetime(r_gcc['Dates_YYYYMMDD'])
l_gcc['Date'] = pd.to_datetime(l_gcc['Dates_YYYYMMDD'])

# Sort chronologically
r_gcc = r_gcc.sort_values('Date').reset_index(drop=True)
l_gcc = l_gcc.sort_values('Date').reset_index(drop=True)

# Find common date range
start_date = max(l_gcc['Date'].min(), r_gcc['Date'].min())
end_date = min(l_gcc['Date'].max(), r_gcc['Date'].max())

# Get dates
image_dates = pd.to_datetime(dates_labs)

# Restrict both datasets to common period
l_plot = l_gcc[
    (l_gcc['Date'] >= start_date) &
    (l_gcc['Date'] <= end_date)
].copy()

r_plot = r_gcc[
    (r_gcc['Date'] >= start_date) &
    (r_gcc['Date'] <= end_date)
].copy()

# Number of animation frames
nframes = max(len(l_plot), len(r_plot))

In [ ]:
# Create figure
fig, ax = plt.subplots(figsize=(12, 7))

# Set y-axis limits based on both datasets
ymin = min(l_plot['Green_Index_av'].min(), r_plot['Green_Index_av'].min())
ymax = max(l_plot['Green_Index_av'].max(), r_plot['Green_Index_av'].max())
# Add a little padding
ypad = (ymax - ymin) * 0.05
ax.set_xlim(start_date, end_date)
ax.set_ylim(ymin - ypad, ymax + ypad)
# Axis labels and title
ax.set_xlabel('Date')
ax.set_ylabel('Greenness')
ax.set_title(f'{house} - {picam}\n''Daily Mean Greenness')
# Create empty lines
line_l, = ax.plot([], [], linewidth=2, label=plotL_treatment)
line_r, = ax.plot([], [], linewidth=2, label=plotR_treatment)
# Add legend
ax.legend()
# Grid
ax.grid(True, alpha=0.3)
# Rotate date labels
fig.autofmt_xdate()

def init():
    """Initialize empty animation."""
    line_l.set_data([], [])
    line_r.set_data([], [])
    return line_l, line_r

def update(frame):
    """Update both GCC timeseries for each animation frame."""
    
    # Left plot
    if frame < len(l_plot):
        line_l.set_data(
            l_plot['Date'].iloc[:frame + 1],
            l_plot['Green_Index_av'].iloc[:frame + 1]
        )

    # Right plot
    if frame < len(r_plot):
        line_r.set_data(
            r_plot['Date'].iloc[:frame + 1],
            r_plot['Green_Index_av'].iloc[:frame + 1]
        )

    # Update title with current date
    if frame < len(l_plot):
        current_date = l_plot['Date'].iloc[frame]
    else:
        current_date = r_plot['Date'].iloc[frame]

    ax.set_title(
        f'{house} - {picam}\n'
        f'Daily Mean Vegetation Greenness — {current_date:%Y-%m-%d}'
    )

    return line_l, line_r

# Create animation
anim = FuncAnimation(
    fig,
    update,
    init_func=init,
    frames=nframes,
    interval=100,
    blit=True,
    repeat=True
)

plt.close(fig)

# Display animation in Jupyter
HTML(anim.to_jshtml())

In [ ]:
# # Output filename
# gcc_gif_filename = (house + '_' +picam +'_Green_Index_Timeseries.gif')
# gcc_gif_filepath = os.path.join(gif_fp, gcc_gif_filename)

# # Save animation
# anim.save(gcc_gif_filepath, writer = 'pillow', fps=10)

# print(f'Saved animation to:\n{gcc_gif_filepath}')

#### 9) Combine PiCam image timelapse and GCC timeseries animations on single figure

In [ ]:
# Read the GIF that was already created
gif = Image.open(gif_filepath)

# Extract all GIF frames
gif_frames = []

try:
    while True:
        # Convert to RGB so matplotlib can display it
        frame = gif.convert('RGB').copy()
        gif_frames.append(np.array(frame))

        gif.seek(gif.tell() + 1)

except EOFError:
    pass

gif.close()

print(f'Number of GIF frames: {len(gif_frames)}')

In [ ]:
# Create dataframe corresponding to GIF frames
gif_dates = pd.DataFrame({
    'Date': image_dates[:len(gif_frames)]
})

print(f'GIF frames: {len(gif_frames)}')
print(f'GIF dates:  {len(gif_dates)}')

In [ ]:
# Match GCC observations to the GIF dates
gif_dates = gif_dates.sort_values('Date').reset_index(drop=True)

# Match left GCC
gif_dates = pd.merge_asof(
    gif_dates,
    l_gcc[['Date', 'Green_Index_av']].sort_values('Date'),
    on='Date',
    direction='nearest'
)

gif_dates = gif_dates.rename(columns={'Green_Index_av': 'S1'})

# Match right GCC
gif_dates = pd.merge_asof(
    gif_dates,
    r_gcc[['Date', 'Green_Index_av']].sort_values('Date'),
    on='Date',
    direction='nearest'
)

gif_dates = gif_dates.rename(columns={'Green_Index_av': 'S4'})

print(gif_dates.head())

In [ ]:
# Create the combined image + GCC figure
fig = plt.figure(figsize=(14, 12))

gs = fig.add_gridspec(
    2, 1,
    height_ratios=[3, 1.3],   # image panel larger than GCC panel
    hspace=0.08               # small gap between panels
)

ax_img = fig.add_subplot(gs[0])
ax_gcc = fig.add_subplot(gs[1])

# ---------------------------------------------------------
# TOP: IMAGE TIMELAPSE
# ---------------------------------------------------------

# ax_img = fig.add_axes(
#     [0.05, 0.40, 0.90, 0.56]
# )

image_artist = ax_img.imshow(gif_frames[0])
ax_img.axis('off')
ax_img.margins(0)

# ---------------------------------------------------------
# BOTTOM: GCC TIMESERIES
# ---------------------------------------------------------

# ax_gcc = fig.add_axes(
#     [0.10, 0.08, 0.82, 0.25]
# )

# GCC y-axis limits
gcc_min = min(
    gif_dates['S4'].min(),
    gif_dates['S1'].min()
)

gcc_max = max(
    gif_dates['S4'].max(),
    gif_dates['S1'].max()
)

gcc_range = gcc_max - gcc_min

# Handle case where values are identical
if gcc_range == 0:
    gcc_range = 1

gcc_padding = gcc_range * 0.05

ax_gcc.set_xlim(gif_dates['Date'].min(), gif_dates['Date'].max())
ax_gcc.set_ylim(gcc_min - gcc_padding, gcc_max + gcc_padding)

# ---------------------------------------------------------
# EMPTY GCC LINES
# ---------------------------------------------------------

line_s4, = ax_gcc.plot(
    [],
    [],
    linewidth=2,
    label='S4'
)

line_s1, = ax_gcc.plot(
    [],
    [],
    linewidth=2,
    label='S1'
)

# ---------------------------------------------------------
# CURRENT DATE INDICATOR
# ---------------------------------------------------------

date_line = ax_gcc.axvline(
    gif_dates['Date'].iloc[0],
    linestyle='--',
    linewidth=1.5
)

# ---------------------------------------------------------
# GCC AXIS FORMATTING
# ---------------------------------------------------------

ax_gcc.set_xlabel('')
ax_gcc.set_ylabel('Vegetation Greenness (%)')
ax_gcc.xaxis.set_major_formatter(DateFormatter('%Y-%m-%d'))
ax_gcc.legend(loc = 'upper right', fontsize = 15)
ax_gcc.grid(True, alpha=0.3)
ax_gcc.margins(x=0.01, y=0.05)

# Rotate date labels
# plt.setp(
#     ax_gcc.get_xticklabels(),
#     rotation=45,
#     ha='right'
# )

# ---------------------------------------------------------
# Reduce the overall figure margins
# ---------------------------------------------------------

fig.subplots_adjust(
    left = 0.06,
    right = 0.96,
    top = 0.98,
    bottom = 0.05
)

# ---------------------------------------------------------
# ANIMATION UPDATE FUNCTION
# ---------------------------------------------------------

def update(frame):

    # ---------------------------------------------
    # Get current GIF frame
    # ---------------------------------------------

    image_artist.set_data(gif_frames[frame])

    # ---------------------------------------------
    # Get GCC data through current date
    # ---------------------------------------------

    current_data = gif_dates.iloc[:frame + 1]
    current_date = gif_dates['Date'].iloc[frame]

    # ---------------------------------------------
    # Update S4
    # ---------------------------------------------

    line_s4.set_data(
        current_data['Date'],
        current_data['S4']
    )

    # ---------------------------------------------
    # Update S1
    # ---------------------------------------------

    line_s1.set_data(
        current_data['Date'],
        current_data['S1']
    )

    # ---------------------------------------------
    # Move vertical date line
    # ---------------------------------------------

    date_line.set_xdata([current_date, current_date])

    # ---------------------------------------------
    # Update title
    # ---------------------------------------------

    # ax_img.set_title(
    #     f'{house} - {picam}\n'
    #     f'{current_date:%Y-%m-%d}',
    #     fontsize=16
    # )

    return (
        image_artist,
        line_s4,
        line_s1,
        date_line
    )

# ---------------------------------------------------------
# CREATE ANIMATION
# ---------------------------------------------------------

anim = FuncAnimation(
    fig,
    update,
    frames=len(gif_frames),
    interval=250,
    repeat=True,
    blit=False
)

plt.close(fig)

# Display in Jupyter
HTML(anim.to_jshtml())

In [ ]:
# Specify output dimensions
gif_width = 1300
gif_height = 1000
gif_dpi = 100

# Set figure dimensions to match desired pixel dimensions
fig.set_size_inches(gif_width / gif_dpi, gif_height / gif_dpi)

# Set filename/filepath
combined_gif_filename = (house + '_' + picam + '_Image_GCC_Combined.gif')
combined_gif_filepath = os.path.join(gif_fp, combined_gif_filename)

# Save the combined animation as a new GIF
anim.save(combined_gif_filepath, writer = PillowWriter(fps = 4))
print(f'Combined GIF saved to:\n'f'{combined_gif_filepath}')